# fastsae `hb_area()` engines on an NVIDIA GPU (Kaggle)

This notebook measures `hb_area()` with **INLA**, **JAX on the CPU** and **JAX on the CUDA GPU** (plus
**tipsae**, Stan, for the Beta model) on spatial (Besag ICAR) area-level models. The results feed the
pkgdown article
[hb_area() engines](https://ridsonap.github.io/fastsae/articles/hb-area-engines.html).

**How to run**

1. In Kaggle: *Create → New Notebook → File → Import Notebook*, then paste the GitHub URL of this file
   (`https://github.com/ridsonap/fastsae/blob/main/benchmarks/kaggle/hb_area_engines_kaggle.ipynb`),
   or upload the `.ipynb`.
2. In the right-hand panel, set *Session options*:
   - **Language: R**.
   - **Accelerator: GPU T4 x2** (or GPU P100).
   - **Internet: On**. This needs a phone-verified Kaggle account.
3. Click **Save Version → Save & Run All (Commit)**. Kaggle runs the whole notebook on its servers,
   for up to 12 hours, without the browser open. Expect about 3–5 hours.
4. When the version is complete, open it and go to the **Output** tab. Download
   `hb_engine_benchmark_kaggle.rds` (and `bench.log`).

In [ ]:
# 1. Check the GPU
cat(system("nvidia-smi", intern = TRUE), sep = "\n")

In [ ]:
# 2. R packages: binaries from Posit Package Manager, INLA from its own repository, fastsae from GitHub
os <- readLines("/etc/os-release")
codename <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
options(
  repos = c(P3M = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename),
            CRAN = "https://cloud.r-project.org"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
  Ncpus = parallel::detectCores()
)
have <- function(p) requireNamespace(p, quietly = TRUE)
need <- c("remotes", "sf", "spdep", "reticulate", "ggplot2", "rstan")
if (length(missing <- need[!vapply(need, have, logical(1))]) > 0) install.packages(missing)
if (!have("sf")) {
  # sf needs the GDAL/GEOS/PROJ system libraries
  invisible(system("apt-get -qq update && apt-get -qq install -y libudunits2-dev libgdal-dev libgeos-dev libproj-dev",
                   intern = TRUE))
  install.packages(c("sf", "spdep"))
}
if (!have("INLA")) {
  install.packages("INLA", repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"),
                   dependencies = TRUE)
}
# tipsae separately, keeping its build log: if it fails, the benchmark runs without it
if (!have("tipsae")) install.packages("tipsae", keep_outputs = TRUE)
if (!have("tipsae")) {
  message("tipsae could not be installed; the benchmark will skip it. Last lines of its log:")
  if (file.exists("tipsae.out")) cat(tail(readLines("tipsae.out"), 40), sep = "\n")
}
remotes::install_github("ridsonap/fastsae", upgrade = "never", force = TRUE)
sapply(c("fastsae", "INLA", "sf", "spdep", "tipsae", "reticulate"),
       function(p) if (have(p)) as.character(packageVersion(p)) else NA)

In [ ]:
# 3. Python side: JAX with CUDA 12 and NumPyro, in the virtualenv "r-numpyro-gpu"
Sys.unsetenv("RETICULATE_PYTHON")  # use the virtualenv, not a preconfigured Python
ok <- tryCatch({ fastsae::setup_numpyro_env(device = "cuda"); TRUE },
               error = function(e) { message(conditionMessage(e)); FALSE })
if (!ok) {
  # virtualenv creation needs the venv module
  invisible(system("apt-get -qq update && apt-get -qq install -y python3-venv", intern = TRUE))
  fastsae::setup_numpyro_env(device = "cuda")
}
# Check in a fresh R process (JAX fixes its device per process): it must list a CUDA device
cat(system("Rscript -e 'fastsae::check_numpyro_available(device = \"cuda\")' 2>&1", intern = TRUE), sep = "\n")

In [ ]:
# 4. Benchmark (INLA, JAX CPU, JAX CUDA, tipsae if installed). Runs in the foreground:
#    in "Save & Run All" mode the session ends when the last cell finishes.
download.file("https://raw.githubusercontent.com/ridsonap/fastsae/main/benchmarks/bench_hb_area_engines.R",
              "bench_hb_area_engines.R", quiet = TRUE)
Sys.setenv(RETICULATE_PYTHON = reticulate::virtualenv_python("r-numpyro-gpu"))
out <- "/kaggle/working/hb_engine_benchmark_kaggle.rds"
engines <- c("inla", "cpu", "cuda", if (requireNamespace("tipsae", quietly = TRUE)) "tipsae")
t0 <- Sys.time()
status <- system2("Rscript", c("bench_hb_area_engines.R", paste0("--engines=", paste(engines, collapse = ",")),
                               paste0("--out=", out)),
                  stdout = "/kaggle/working/bench.log", stderr = "/kaggle/working/bench.err")
cat(grep(" n = |Error|Saved|failed", readLines("/kaggle/working/bench.log"), value = TRUE), sep = "\n")
if (status != 0) cat(tail(readLines("/kaggle/working/bench.err"), 30), sep = "\n")
cat("\nExit status:", status, " Elapsed:", format(round(difftime(Sys.time(), t0, units = "hours"), 2)), "\n")

In [ ]:
# 5. Results (also kept in the saved version of this notebook)
res <- readRDS(out)
res$meta[c("machine", "cpu", "cpu_cores", "gpu", "ram_gb", "python")]
res$timings
res$agreement

**6. Download:** open the saved version → **Output** tab → download `hb_engine_benchmark_kaggle.rds`.
This file is added to the article as `inst/extdata/hb_engine_benchmark_kaggle_<gpu>.rds`.